# `detour_factor` sweep for `DeviatedFixedRoutePolicy`

`DeviatedFixedRoutePolicy` keeps the skip-stop backbone (which caps ride time
~for free via the cycle) and adds **bounded detours** to grab nearby out-of-order
demand. `detour_factor` is the knob: `1.0` reduces exactly to skip-stop; larger
values allow bigger detours, trading a slowly-rising ride tail for better wait
and coverage. Because total deviation per cycle is capped at
`(detour_factor − 1) × cycle`, a rider's ride stays within `detour_factor ×` the
base cycle — so this sweep should hug the **low-max-ride** region, unlike the
`ride_factor` sweep which started from greedy's blown-up ride tail.

Each swept value is its own column, run on the **same demand seeds** as the
baselines. `compare_policies` renders one **tqdm** bar (with ETA) per column.

**What we're testing:** does this frontier *dominate* the RideBounded one — i.e.
reach served ≥ skip-stop's ~72% (ideally toward Nearest's ~74.6%) and mean wait
below skip-stop's ~461s, while **max ride stays near skip-stop's ~1,300s**?

In [ ]:
%matplotlib inline

import statistics

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from src.algorithm.FixedRoutePolicy import FixedRoutePolicy
from src.algorithm.SkipStopFixedRoutePolicy import SkipStopFixedRoutePolicy
from src.algorithm.NearestUnservedPolicy import NearestUnservedPolicy
from src.algorithm.DeviatedFixedRoutePolicy import DeviatedFixedRoutePolicy
from src.experiment import summarize_comparison
from src.map.PlanarMapProvider import PlanarMapProvider
from src.map.graph import PlanarStreetGraph

In [ ]:
MAP_SEED = 42

graph = PlanarStreetGraph.generate(
    n_seeds=17,
    bounds=(0.0, 0.0, 2000.0, 2000.0),
    seed=MAP_SEED,
    stop_density=0.5,
)

provider = PlanarMapProvider(graph)
print(f"{len(provider.get_stops())} bus stops on this map")

In [ ]:
# Swept detour_factor values (edit freely). detour_factor=1.0 == skip-stop,
# which is already a baseline column, so the sweep starts above 1.0.
DETOUR_FACTORS = [1.5, 2.0, 2.5, 3.0]

N_TRIALS = 1000
BASE_SEED = 0

# run_trial kwargs, matching the campus experiment.
TRIAL_KWARGS = dict(max_requests=200, capacity=40, max_wait_s=900.0)

# Baselines (skip-stop = ride-cap anchor, Nearest = coverage anchor) plus one
# DeviatedFixedRoute column per swept factor. The `df=df` default-arg capture
# avoids the late-binding closure bug.
policies = [
    ("Fixed route", lambda g: FixedRoutePolicy(g)),
    ("Fixed route w/ skip", lambda g: SkipStopFixedRoutePolicy(g)),
    ("Nearest unserved", lambda g: NearestUnservedPolicy(g)),
] + [
    (
        f"DFR df={df}",
        lambda g, df=df: DeviatedFixedRoutePolicy(g, detour_factor=df),
    )
    for df in DETOUR_FACTORS
]

summary = summarize_comparison(
    policies=policies,
    graph=graph,
    n_trials=N_TRIALS,
    base_seed=BASE_SEED,
    **TRIAL_KWARGS,
)

display(Markdown(summary.markdown))

In [ ]:
# Frontier: mean served fraction (x) vs mean per-trial max ride (y) per policy.
def _mean_served(trials):
    values = [
        t.kpi.served_fraction
        for t in trials
        if t.kpi.served_fraction is not None
    ]
    return statistics.fmean(values) if values else None


def _mean_max_ride(trials):
    per_trial_max = [max(t.ride_times) for t in trials if t.ride_times]
    return statistics.fmean(per_trial_max) if per_trial_max else None


points = {
    label: (_mean_served(trials), _mean_max_ride(trials))
    for label, trials in summary.results.items()
}
points = {
    label: (served, max_ride)
    for label, (served, max_ride) in points.items()
    if served is not None and max_ride is not None
}

dfr_labels = [f"DFR df={df}" for df in DETOUR_FACTORS if f"DFR df={df}" in points]
baseline_labels = [label for label in points if label not in dfr_labels]

fig, ax = plt.subplots(figsize=(8, 6))

# Deviated-route frontier: connect points in detour_factor order.
dfr_x = [points[label][0] * 100.0 for label in dfr_labels]
dfr_y = [points[label][1] for label in dfr_labels]
ax.plot(dfr_x, dfr_y, "-o", color="tab:green", label="Deviated route (by df)")

# Baselines as separate markers.
for label in baseline_labels:
    served, max_ride = points[label]
    ax.scatter(served * 100.0, max_ride, marker="s", color="tab:orange", zorder=3)

for label, (served, max_ride) in points.items():
    ax.annotate(
        label,
        (served * 100.0, max_ride),
        textcoords="offset points",
        xytext=(6, 6),
        fontsize=9,
    )

ax.set_xlabel("Served fraction (%)")
ax.set_ylabel("Mean per-trial max ride time (s)")
ax.set_title("Served fraction vs max ride — detour_factor frontier")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## Reading the frontier

- **The win condition:** the green `DFR df=…` points should sit **low** (near
  skip-stop's max ride, far below the RideBounded frontier's 4,000–10,000s) while
  pushing **right** (higher served fraction) and — check the table — cutting mean
  wait below skip-stop's ~461s. A DFR point that is at or right of skip-stop on
  served *and* below it on wait, with only a modest max-ride increase, is a
  strict improvement over skip-stop and the first policy to genuinely beat the
  skip-vs-Nearest front.
- **If the green points barely move right of skip-stop**, detours aren't finding
  much reachable out-of-order demand at this scale/geometry — the backbone
  already visits stops in near-optimal order (a real possibility on a compact
  campus map), and skip-stop stands as the answer.
- **If max ride climbs steeply with detour_factor**, tighten the knob (smaller
  detour_factor) — the value is in the low-df region that keeps rides bounded.
- Compare directly against the `ride_factor` sweep: the whole thesis is that
  attacking from the fixed-route side lands in the low-max-ride corner that
  RideBounded could never reach.